In [1]:
import time
import math

import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from datasets import load_dataset
import tiktoken
import itertools

torch.backends.cudnn.benchmark = True

# Softmax and angular attention

In [2]:
# =========================================================================
# 1. Attention‐block definitions for LMModel
# =========================================================================
# A. Softmax MHA
class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.att  = MultiHeadAttention(
            d_in=cfg["emb_dim"], d_out=cfg["emb_dim"],
            context_length=cfg["context_length"],
            dropout=cfg["drop_rate"], num_heads=cfg["n_heads"],
            qkv_bias=cfg["qkv_bias"]
        )
        self.norm1 = nn.LayerNorm(cfg["emb_dim"])
        self.norm2 = nn.LayerNorm(cfg["emb_dim"])
        self.ff    = nn.Sequential(
                        nn.Linear(cfg["emb_dim"],4*cfg["emb_dim"]),
                        nn.GELU(),
                        nn.Linear(4*cfg["emb_dim"],cfg["emb_dim"])
                     )
        self.drop  = nn.Dropout(cfg["drop_rate"])

    def forward(self, x):
        h = x
        x = self.norm1(x)
        x = self.att(x); x = self.drop(x) + h
        h = x
        x = self.norm2(x)
        x = self.ff(x); x = self.drop(x) + h
        return x

class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        assert d_out % num_heads == 0
        self.num_heads = num_heads
        self.head_dim  = d_out // num_heads

        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key   = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj= nn.Linear(d_out, d_out)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer("mask",
            torch.triu(torch.ones(context_length, context_length), diagonal=1).bool()
        )


    def forward(self, x):
        B, T, _ = x.shape
        Q = self.W_query(x).view(B, T, self.num_heads, self.head_dim).transpose(1,2)
        K = self.W_key(x).view(B, T, self.num_heads, self.head_dim).transpose(1,2)
        V = self.W_value(x).view(B, T, self.num_heads, self.head_dim).transpose(1,2)

        scores = (Q @ K.transpose(-2,-1)) / math.sqrt(self.head_dim)
        scores = scores.masked_fill(self.mask[:T,:T], float('-inf'))
        W = torch.softmax(scores, dim=-1)
        W = self.dropout(W)

        out = (W @ V).transpose(1,2).reshape(B, T, -1)
        return self.out_proj(out)

# B. Angular
class AngularBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.att  = AngularAttention(
            d_in=cfg["emb_dim"], d_out=cfg["emb_dim"],
            context_length=cfg["context_length"],
            dropout=cfg["drop_rate"], num_heads=cfg["n_heads"],
            qkv_bias=cfg["qkv_bias"]
        )
        self.norm1 = nn.LayerNorm(cfg["emb_dim"])
        self.norm2 = nn.LayerNorm(cfg["emb_dim"])
        self.ff    = nn.Sequential(
                        nn.Linear(cfg["emb_dim"],4*cfg["emb_dim"]),
                        nn.GELU(),
                        nn.Linear(4*cfg["emb_dim"],cfg["emb_dim"])
                     )
        self.drop  = nn.Dropout(cfg["drop_rate"])

    def forward(self, x):
        h = x
        x = self.norm1(x)
        x = self.att(x); x = self.drop(x) + h
        h = x
        x = self.norm2(x)
        x = self.ff(x); x = self.drop(x) + h
        return x

class AngularAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        assert d_out % num_heads == 0
        self.num_heads = num_heads
        self.head_dim  = d_out // num_heads

        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key   = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj= nn.Linear(d_out, d_out)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer("mask",
            torch.triu(torch.ones(context_length, context_length), diagonal=1).bool()
        )

    def forward(self, x):
        B, T, _ = x.shape
        Q = self.W_query(x).view(B,T,self.num_heads,self.head_dim).transpose(1,2)
        K = self.W_key(x).view(B,T,self.num_heads,self.head_dim).transpose(1,2)
        V = self.W_value(x).view(B,T,self.num_heads,self.head_dim).transpose(1,2)

        Q = F.normalize(Q, dim=-1, eps=1e-6)
        K = F.normalize(K, dim=-1, eps=1e-6)

        cos_sim = (Q @ K.transpose(-2,-1)).clamp(-0.999,0.999)
        scores  = 1 - torch.acos(cos_sim)/torch.pi
        scores.masked_fill_(self.mask[:T,:T], 0.0)
        W = scores.clamp(min=1e-6).pow(18.0)
        W = W / (W.sum(-1,keepdim=True)+1e-6)
        W = self.dropout(W)

        out = (W @ V).transpose(1,2).reshape(B, T, -1)
        return self.out_proj(out)

# Non-differentiable RACEAttention with no causal masking: DO NOT RUN THIS!
* This is the older approach, much worse than vanilla and angular attention because of nonsmooth bucketing

In [ ]:
# C. RACE

#=====================================================================
# Fast, loop‑free ACE sketch works for S = H.N_M heads
#=====================================================================
class BatchedACE(nn.Module):
    """
    planes_flat : [S, L, K, d_k]   (float16)
    A_flat      : [S*L, R]         (int16 counters)
    B_flat      : [S*L, R, d_k]    (float16 value sums)

    In some part of the code, I called S as NH.

    This is the non-differentiable RACE
    """
    def __init__(self, d_k, K, L, N_M, H, device, alpha=0.25):
        super().__init__()
        self.K, self.L   = K, L
        self.N_M, self.H = N_M, H
        self.d_k, self.R = d_k, 1 << K
        self.alpha       = alpha

        NH  = H * N_M         # total (ensemble × head)
        HL  = NH * L          # flattened rows

        # 1. random hyperplanes  … store as float16 to cut BW
        planes = torch.randn(N_M, H, L, K, d_k, device=device).half()
        self.register_buffer("planes_flat", planes.reshape(NH, L, K, d_k))

        # 2. bit‑weights  [1,1,1,K]  for packing LSH bits → bucket id
        bw = (2 ** torch.arange(K, device=device)).view(1, 1, 1, K)
        self.register_buffer("bit_weights", bw)

        # 3. flattened counter / sum buffers
        self.register_buffer("A_flat",
                             torch.zeros(HL, self.R, dtype=torch.int16, device=device))
        self.register_buffer("B_flat",
                             torch.zeros(HL, self.R, d_k, dtype=torch.float16, device=device))

    # ----------------------------------------------------------------------
    @torch.no_grad()
    def clear(self):
        self.A_flat.zero_()
        self.B_flat.zero_()

    # ----------------------------------------------------------------------
    @torch.no_grad()
    def add_batch(self, Khf, Vhf):
        """
        Khf, Vhf : [S, N, d_k]   (S = H.N_M ,  N = B·T)
        """
        NH, N, _ = Khf.shape
        L, K, d_k = self.L, self.K, self.d_k
        HL = NH * L

        # 1. hash every token once  --> buckets [NH, N, L]
        proj    = torch.einsum('pnd,plkd->pnlk', Khf.half(), self.planes_flat)
        bits    = proj > 0
        buckets = (bits * self.bit_weights).sum(-1).long()          # [NH,N,L]

        # 2. stochastic token subsample (alpha fraction kept)
        keep = torch.rand(N, device=Khf.device) < self.alpha        # [N]
        buckets   = buckets[:, keep]                                # [NH,N_sub,L]
        Vhf_sub   = Vhf[:, keep].half()                             # [NH,N_sub,d_k]
        N_sub     = buckets.size(1)

        # 3. flatten (NH,L) --> HL and scatter once
        idx_flat = buckets.permute(0, 2, 1).reshape(HL, N_sub)      # [HL,N_sub]

        #   counters
        self.A_flat.scatter_add_(
            1, idx_flat,
            torch.ones_like(idx_flat, dtype=self.A_flat.dtype)
        )

        #   value sums
        Vflat = (Vhf_sub.unsqueeze(1)                               # [NH,1,N_sub,d_k]
                           .expand(NH, L, N_sub, d_k)
                           .reshape(HL, N_sub, d_k))                # [HL,N_sub,d_k]
        self.B_flat.scatter_add_(
            1,
            idx_flat.unsqueeze(-1).expand(-1, -1, d_k),             # [HL,N_sub,d_k]
            Vflat
        )

    # ----------------------------------------------------------------------
    def query(self, Qhf, row_choice: int = 0):
        """
        Qhf : [NH, N, d_k]    -->   returns [NH, N, d_k]
        Fast path: estimate with **one** sketch row (row_choice).
        """
        NH, N, _ = Qhf.shape
        row = max(0, min(row_choice, self.L - 1))                   # clamp

        # 1) hash queries once  → buckets_row [NH,N]
        proj = torch.einsum('pnd,plkd->pnlk', Qhf.half(), self.planes_flat)
        bits = proj > 0
        buckets_row = (bits * self.bit_weights).sum(-1)[:, :, row].long()

        # 2) gather counts & sums for that row
        #    self.A_flat row slice is [NH, R] thanks to flattening:
        row_offset = row + torch.arange(NH, device=Qhf.device) * self.L
        A_row = self.A_flat[row_offset]               # [NH, R]
        B_row = self.B_flat[row_offset]               # [NH, R, d_k]

        cnt = A_row.gather(1, buckets_row)            # [NH, N]
        val = B_row.gather(
                1,
                buckets_row.unsqueeze(-1).expand(-1, -1, self.d_k)
              )                                       # [NH, N, d_k]

        return val.float() / (cnt.unsqueeze(-1).float() + 1e-6)


#=====================================================================
# RACEAttention with the new BatchedACE
#=====================================================================
class RACEAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout,
                 num_heads, qkv_bias=False, L=2, K=3, N_M=1,
                 alpha=0.25, device='cuda'):
        super().__init__()
        assert d_in % num_heads == 0
        self.H   = num_heads
        self.d_k = d_in // num_heads
        self.N_M = N_M

        # Linear projections
        self.q_proj = nn.Linear(d_in, d_in, bias=qkv_bias)
        self.k_proj = nn.Linear(d_in, d_in, bias=qkv_bias)
        self.v_proj = nn.Linear(d_in, d_in, bias=qkv_bias)
        self.out    = nn.Linear(d_in, d_out)
        self.dropout = nn.Dropout(dropout)

        # Fused ACE sketch
        self.ace = BatchedACE(
            d_k=self.d_k, K=K, L=L, N_M=N_M, H=num_heads,
            alpha=alpha, device=device
        )

    # ------------------------------------------------------------------
    def forward(self, x):
        B, T, _ = x.shape
        N = B * T

        # project & reshape → [B,T,H,d_k]
        Q = self.q_proj(x).view(B, T, self.H, self.d_k)
        K = self.k_proj(x).view(B, T, self.H, self.d_k)
        V = self.v_proj(x).view(B, T, self.H, self.d_k)

        # collapse heads --> [H,N,d_k]   then tile N_M ensembles
        def collapse(t):
            return t.permute(2, 0, 1, 3).reshape(self.H, N, self.d_k)
        Khf = collapse(K).repeat(self.N_M, 1, 1)
        Vhf = collapse(V).repeat(self.N_M, 1, 1)
        Qhf = collapse(Q).repeat(self.N_M, 1, 1)

        # build + query sketch
        with torch.no_grad():
            self.ace.clear()
            self.ace.add_batch(Khf, Vhf)
            
        out = self.ace.query(Qhf)                     # [NH,N,d_k]

        # reshape back → [B,T,H,d_k] then merge heads
        out = out.view(self.N_M, self.H, B, T, self.d_k).mean(0)  # [H,B,T,d_k]
        out = out.permute(1, 2, 0, 3).reshape(B, T, -1)           # [B,T,d_in]

        return self.dropout(self.out(out))



class RACEBlock(nn.Module):
    
    def __init__(self, cfg, device):
        super().__init__()
        self.att = RACEAttention(
            d_in=cfg["emb_dim"], d_out=cfg["emb_dim"],
            context_length=cfg["context_length"],
            dropout=cfg["drop_rate"], num_heads=cfg["n_heads"],
            qkv_bias=cfg["qkv_bias"], 
            L=2, K=3, N_M=2, alpha=0.1, device=device
        )
        self.norm1 = nn.LayerNorm(cfg["emb_dim"])
        self.norm2 = nn.LayerNorm(cfg["emb_dim"])
        self.ff    = nn.Sequential(
                        nn.Linear(cfg["emb_dim"], 4*cfg["emb_dim"]),
                        nn.GELU(),
                        nn.Linear(4*cfg["emb_dim"], cfg["emb_dim"])
                     )
        self.drop  = nn.Dropout(cfg["drop_rate"])

    def forward(self, x):
        h = x
        x = self.norm1(x)
        x = self.att(x); x = self.drop(x) + h

        h = x
        x = self.norm2(x)
        x = self.ff(x); x = self.drop(x) + h
        return x

# Learned RACE: USE THIS!
* No half-precision for the time being
* Soft (differentiable) bucketization
* No sampling parameter $\alpha$ at this point

In [6]:
class BatchedACE(nn.Module):
    def __init__(self, d_k, K, L, N_M, H, device='cuda'):
        super().__init__()
        self.K, self.L, self.d_k = K, L, d_k
        self.R = 1 << K

        # full-precision planes
        planes = torch.randn(L, K, d_k, device=device)
        self.register_buffer('planes_flat', planes)

        # bucket prototypes
        corners = torch.tensor(
            list(itertools.product([-1.0, +1.0], repeat=K)),
            device=device
        )
        self.register_buffer('bucket_protos', corners)

        # # learnable temperature
        # self.logit_temp = nn.Parameter(torch.log(torch.tensor(1.0)))

    def forward(self, Khf, Vhf, Qhf):
        # Khf, Vhf, Qhf: [M, B, T, H, d_k]
        # we assume they are already shaped as [M,B,T,H,d_k]
        M, B, T, H, d_k = Khf.shape
        L, K, _ = self.planes_flat.shape

        # temp is basically the scaling factor in attention. One can make it learnable
        # to further improve the performance
        temp = math.sqrt(d_k)
        # temp = self.logit_temp.exp().clamp(1e-2, 10.0) # uncomment when you make temp learnable
        

        # 1) soft-hash keys --> [M,B,T,H,L,R]
        proj_K   = torch.einsum('mbthd,lkd->mbthlk', Khf, self.planes_flat)
        softbits = torch.tanh(proj_K.clamp(-10,10)) / temp
        logits_K = torch.einsum('mbthlk,rk->mbthlr', softbits, self.bucket_protos)
        probs_K  = torch.softmax(logits_K, dim=-1)

        # 2) causal counts & sums
        V_ext  = Vhf.unsqueeze(-2).unsqueeze(-2)  # [M,B,T,H,1,1,d_k]
        A_pref = probs_K.cumsum(dim=2)            # prefix over T
        B_pref = (probs_K.unsqueeze(-1) * V_ext).cumsum(dim=2)
        E_pref = B_pref / (A_pref.unsqueeze(-1) + 1e-6)

        # 3) soft-hash queries --> [M,B,T,H,L,R]
        proj_Q   = torch.einsum('mbthd,lkd->mbthlk', Qhf, self.planes_flat)
        bits_Q   = torch.tanh(proj_Q.clamp(-10,10)) / temp
        logits_Q = torch.einsum('mbthlk,rk->mbthlr', bits_Q, self.bucket_protos)
        probs_Q  = torch.softmax(logits_Q, dim=-1)

        # 4) expected‐value lookup --> [M,B,T,H,d_k]
        mul      = probs_Q.unsqueeze(-1) * E_pref    # [M,B,T,H,L,R,d_k]
        sum_L    = mul.sum(dim=4)                    # sum over L --> [M,B,T,H,R,d_k]
        out_pref = sum_L.sum(dim=4)                  # sum over R --> [M,B,T,H,d_k]

        return out_pref  # [M,B,T,H,d_k]

class RACEAttention(nn.Module):
    def __init__(self, d_in, d_out, dropout,
                 num_heads, qkv_bias=False, L=2, K=3, N_M=1, device='cuda'):
        super().__init__()
        assert d_in % num_heads == 0
        self.H   = num_heads
        self.d_k = d_in // num_heads
        self.M   = N_M

        self.q_proj = nn.Linear(d_in, d_in, bias=qkv_bias)
        self.k_proj = nn.Linear(d_in, d_in, bias=qkv_bias)
        self.v_proj = nn.Linear(d_in, d_in, bias=qkv_bias)
        self.out    = nn.Linear(d_in, d_out)
        self.drop   = nn.Dropout(dropout)

        self.ace = BatchedACE(self.d_k, K, L, N_M, num_heads, device=device)

    def forward(self, x):
        B, T, _ = x.shape
        H, d_k, M = self.H, self.d_k, self.M

        # 1) project & reshape for ACE
        Q = self.q_proj(x).view(B, T, H, d_k)
        K = self.k_proj(x).view(B, T, H, d_k)
        V = self.v_proj(x).view(B, T, H, d_k)

        # shape --> [M, B, T, H, d_k] by explicit unsqueeze
        def pack(Z):
            Zm = Z.unsqueeze(0).expand(M, -1, -1, -1, -1)
            return Zm

        Khf = pack(K)
        Vhf = pack(V)
        Qhf = pack(Q)

        # 2) run ACE
        out_hm = self.ace(Khf, Vhf, Qhf)  # [M,B,T,H,d_k]

        # 3) average ensembles & merge heads
        out = out_hm.mean(dim=0)          # [B,T,H,d_k]
        out = out.permute(0,2,1,3).reshape(B, T, H * d_k)

        # 4) final proj + dropout
        return self.drop(self.out(out))


class RACEBlock(nn.Module):
    def __init__(self, cfg, device='cuda'):
        super().__init__()
        self.att   = RACEAttention(
            d_in=cfg["emb_dim"], d_out=cfg["emb_dim"],
            dropout=cfg["drop_rate"],
            num_heads=cfg["n_heads"], qkv_bias=cfg["qkv_bias"],
            L=2, K=3, N_M=2, device=device
        )
        self.norm1 = nn.LayerNorm(cfg["emb_dim"])
        self.norm2 = nn.LayerNorm(cfg["emb_dim"])
        self.ff    = nn.Sequential(
            nn.Linear(cfg["emb_dim"], 4*cfg["emb_dim"]),
            nn.GELU(),
            nn.Linear(4*cfg["emb_dim"], cfg["emb_dim"])
        )
        self.drop  = nn.Dropout(cfg["drop_rate"])

    def forward(self, x):
        h = x
        x = self.norm1(x)
        x = self.att(x)
        x = self.drop(x) + h

        h = x
        x = self.norm2(x)
        x = self.ff(x)
        x = self.drop(x) + h

        return x

# LMModel

In [7]:
#=====================================================================
# 2) The unified LMModel
#=====================================================================
class LMModel(nn.Module):
    def __init__(self, cfg, attn_type="gpt", device="cpu"):
        """
        attn_type ∈ {"gpt","angular","race"}
        """
        super().__init__()
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
        self.pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"])
        self.drop_emb= nn.Dropout(cfg["drop_rate"])
        self.final_norm = nn.LayerNorm(cfg["emb_dim"])
        self.out_head   = nn.Linear(cfg["emb_dim"], cfg["vocab_size"], bias=False)

        # choose attention class
        if attn_type == "gpt":
            AttnBlock = TransformerBlock
        elif attn_type == "angular":
            AttnBlock = AngularBlock
        elif attn_type == "race":
            # our custom RACEBlock needs device
            AttnBlock = lambda c: RACEBlock(c, device)
        else:
            raise ValueError(attn_type)

        # build n_layers of whichever block
        self.blocks = nn.Sequential(
            *[AttnBlock(cfg) for _ in range(cfg["n_layers"])]
        )

    def forward(self, x):
        B, T = x.shape
        pos = torch.arange(T, device=x.device).unsqueeze(0)
        x = self.tok_emb(x) + self.pos_emb(pos)
        x = self.drop_emb(x)
        x = self.blocks(x)
        x = self.final_norm(x)
        return self.out_head(x)

# Dataset, training and validation loops

In [8]:
#=====================================================================
# 4) PTB dataset & loader
#=====================================================================
class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, L, S):
        self.inputs, self.targets = [], []
        ids = tokenizer.encode(txt, allowed_special={"<|endoftext|>"})
        for i in range(0, len(ids)-L, S):
            self.inputs .append(torch.tensor(ids[i:i+L]))
            self.targets.append(torch.tensor(ids[i+1:i+L+1]))
    def __len__(self): return len(self.inputs)
    def __getitem__(self,i): return self.inputs[i], self.targets[i]

def create_dataloader(txt, bs, L, S):
    tok = tiktoken.get_encoding("gpt2")
    ds  = GPTDatasetV1(txt, tok, L, S)
    return DataLoader(ds, batch_size=bs, shuffle=True, drop_last=True)

#=====================================================================
# 5) run_experiment
#=====================================================================

def run_experiment(attn_types, epochs=5, lr=1e-5, wd=0.01):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    cfg = {
      "vocab_size":50257, "context_length":64,
      "emb_dim":512, "n_heads":8, "n_layers":4,
      "drop_rate":0.1, "qkv_bias":False
    }


    for kind in attn_types:
        model = LMModel(cfg, attn_type=kind, device=device).to(device)
        opt   = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)


        print(f"\n=== Training {kind.upper()} for {epochs} epochs ===")
        for ep in range(1, epochs+1):
            # --- TR ---
            t0 = time.time()
            model.train()
            tl = ta = 0.0
            for x,y in train_loader:
                opt.zero_grad()
                logits = model(x.to(device))
                loss   = F.cross_entropy(logits.flatten(0,1), y.to(device).flatten())
                acc    = (logits.argmax(-1)==y.to(device)).float().mean().item()
                loss.backward(); 
                opt.step()
                tl += loss.item(); ta += acc
            train_time = time.time() - t0

            tr_l = tl/len(train_loader)
            tr_a = ta/len(train_loader)
            tr_p = math.exp(tr_l)

            # --- VAL---
            t1 = time.time()
            model.eval()
            vl = va = 0.0
            with torch.no_grad():
                for x,y in val_loader:
                    logits = model(x.to(device))
                    loss   = F.cross_entropy(logits.flatten(0,1), y.to(device).flatten())
                    acc    = (logits.argmax(-1)==y.to(device)).float().mean().item()
                    vl += loss.item(); va += acc
            val_time = time.time() - t1

            va_l = vl/len(val_loader)
            va_a = va/len(val_loader)
            va_p = math.exp(va_l)

            print(
                f"Ep{ep:2d} | "
                f"train loss {tr_l:.3f}, acc {tr_a:.3f}, ppl {tr_p:.2f} "
                f"(train_time {train_time:.1f}s) | "
                f"val loss {va_l:.3f}, acc {va_a:.3f}, ppl {va_p:.2f} "
                f"(val_time {val_time:.1f}s)"
            )


# PTB comes with train/validation/test splits, and each row is a sentence.
ptb = load_dataset("ptb_text_only")

# Join all sentences into one long string per split:
tr_txt = " ".join(ptb["train"]["sentence"])
va_txt = " ".join(ptb["validation"]["sentence"])
# use ptb["test"] if you want a held-out test set

# Now create your dataloaders just as before:
train_loader = create_dataloader(tr_txt, bs=16, L=64, S=32)
val_loader   = create_dataloader(va_txt, bs=16, L=64, S=32)

In [6]:
# with standard fixed scaling sqrt(d_k)
run_experiment(["race"],epochs=10)


=== Training RACE for 10 epochs ===
Ep 1 | train loss 6.595, acc 0.169, ppl 731.20 (train_time 215.2s) | val loss 5.784, acc 0.200, ppl 325.21 (val_time 4.4s)
Ep 2 | train loss 5.620, acc 0.216, ppl 275.89 (train_time 213.4s) | val loss 5.447, acc 0.231, ppl 232.11 (val_time 4.4s)
Ep 3 | train loss 5.351, acc 0.237, ppl 210.75 (train_time 213.5s) | val loss 5.247, acc 0.246, ppl 189.96 (val_time 4.4s)
Ep 4 | train loss 5.168, acc 0.250, ppl 175.65 (train_time 214.2s) | val loss 5.103, acc 0.256, ppl 164.58 (val_time 4.4s)
Ep 5 | train loss 5.028, acc 0.259, ppl 152.64 (train_time 214.7s) | val loss 4.995, acc 0.265, ppl 147.69 (val_time 4.4s)
Ep 6 | train loss 4.914, acc 0.267, ppl 136.24 (train_time 214.1s) | val loss 4.911, acc 0.273, ppl 135.79 (val_time 4.4s)
Ep 7 | train loss 4.820, acc 0.274, ppl 123.94 (train_time 214.1s) | val loss 4.848, acc 0.277, ppl 127.49 (val_time 4.4s)
Ep 8 | train loss 4.739, acc 0.280, ppl 114.35 (train_time 214.0s) | val loss 4.788, acc 0.282, ppl 12

In [6]:
run_experiment(["gpt"],epochs=10)


=== Training GPT for 10 epochs ===
Ep 1 | train loss 6.584, acc 0.166, ppl 723.41 (train_time 136.0s) | val loss 5.807, acc 0.198, ppl 332.52 (val_time 3.0s)
Ep 2 | train loss 5.637, acc 0.216, ppl 280.51 (train_time 135.2s) | val loss 5.463, acc 0.228, ppl 235.72 (val_time 3.0s)
Ep 3 | train loss 5.368, acc 0.235, ppl 214.34 (train_time 135.2s) | val loss 5.265, acc 0.242, ppl 193.54 (val_time 3.0s)
Ep 4 | train loss 5.186, acc 0.246, ppl 178.72 (train_time 135.2s) | val loss 5.128, acc 0.252, ppl 168.68 (val_time 3.0s)
Ep 5 | train loss 5.046, acc 0.255, ppl 155.35 (train_time 135.2s) | val loss 5.024, acc 0.259, ppl 152.03 (val_time 3.0s)
Ep 6 | train loss 4.931, acc 0.263, ppl 138.56 (train_time 135.3s) | val loss 4.937, acc 0.266, ppl 139.35 (val_time 3.0s)
Ep 7 | train loss 4.836, acc 0.269, ppl 125.95 (train_time 135.4s) | val loss 4.867, acc 0.272, ppl 129.91 (val_time 3.0s)
Ep 8 | train loss 4.753, acc 0.275, ppl 115.93 (train_time 135.1s) | val loss 4.803, acc 0.277, ppl 121

In [7]:
run_experiment(["angular"],epochs=10)


=== Training ANGULAR for 10 epochs ===
Ep 1 | train loss 6.595, acc 0.165, ppl 731.38 (train_time 152.7s) | val loss 5.795, acc 0.197, ppl 328.73 (val_time 3.3s)
Ep 2 | train loss 5.629, acc 0.215, ppl 278.38 (train_time 152.5s) | val loss 5.459, acc 0.230, ppl 234.98 (val_time 3.3s)
Ep 3 | train loss 5.361, acc 0.236, ppl 212.98 (train_time 152.3s) | val loss 5.256, acc 0.243, ppl 191.79 (val_time 3.3s)
Ep 4 | train loss 5.176, acc 0.248, ppl 176.99 (train_time 152.5s) | val loss 5.113, acc 0.254, ppl 166.15 (val_time 3.3s)
Ep 5 | train loss 5.032, acc 0.257, ppl 153.17 (train_time 152.5s) | val loss 5.001, acc 0.261, ppl 148.63 (val_time 3.3s)
Ep 6 | train loss 4.913, acc 0.265, ppl 136.11 (train_time 152.3s) | val loss 4.909, acc 0.269, ppl 135.52 (val_time 3.3s)
Ep 7 | train loss 4.815, acc 0.273, ppl 123.32 (train_time 152.4s) | val loss 4.836, acc 0.275, ppl 125.94 (val_time 3.3s)
Ep 8 | train loss 4.732, acc 0.279, ppl 113.50 (train_time 152.4s) | val loss 4.777, acc 0.280, ppl

In [9]:
# with learnable scaling
run_experiment(["race"],epochs=10)


=== Training RACE for 10 epochs ===
Ep 1 | train loss 6.588, acc 0.170, ppl 726.46 (train_time 219.4s) | val loss 5.755, acc 0.209, ppl 315.87 (val_time 4.4s)
Ep 2 | train loss 5.484, acc 0.255, ppl 240.74 (train_time 218.9s) | val loss 5.171, acc 0.294, ppl 176.17 (val_time 4.4s)
Ep 3 | train loss 5.001, acc 0.316, ppl 148.58 (train_time 218.5s) | val loss 4.773, acc 0.341, ppl 118.26 (val_time 4.4s)
Ep 4 | train loss 4.654, acc 0.356, ppl 105.05 (train_time 218.7s) | val loss 4.472, acc 0.378, ppl 87.56 (val_time 4.4s)
Ep 5 | train loss 4.379, acc 0.387, ppl 79.77 (train_time 218.5s) | val loss 4.214, acc 0.407, ppl 67.66 (val_time 4.4s)
Ep 6 | train loss 4.137, acc 0.414, ppl 62.61 (train_time 218.4s) | val loss 3.988, acc 0.434, ppl 53.93 (val_time 4.4s)
Ep 7 | train loss 3.923, acc 0.439, ppl 50.55 (train_time 218.6s) | val loss 3.787, acc 0.459, ppl 44.13 (val_time 4.4s)
Ep 8 | train loss 3.729, acc 0.462, ppl 41.64 (train_time 219.4s) | val loss 3.600, acc 0.483, ppl 36.59 (val